In [1]:
import pandas as pd

# Path to data frame with WSIs
df_path = "D:\DATA\with_snomed_category.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to cache file
cache_file = "cache_tissue_artifact_new.pkl"

# Path to zarr
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

Index(['filename', 'rekvnr', 'modtdato', 'rekvprio', 'team', 'sex', 'alder',
       'alder gruppe', 'rekvdato', 'matantal', 'mattype tekst', 'glasalm',
       'immunalm', 'ishutaelalm', 'antglas', 'specfarv', 'stain',
       'snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text',
       'undersoeger_anonymous', 'T_category', 'M_category'],
      dtype='object')


In [2]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

C:\ProgramData\miniconda3\envs\lazyslide_010_py310_uv_2\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
C:\ProgramData\miniconda3\envs\lazyslide_010_py310_uv_2\lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
from helper_functions import subset_df, subset_df_list

df_HE = subset_df(df_all, "stain", "HE")
df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")
df_HE["T_category"].value_counts().head()

T_category
[Female Reproductive System (Maternal)]      16657
[Blood, Bone Marrow and Lymphatic System]     2883
[Placenta, Fetal Membranes, and Fetus]        2627
[Male Reproductive System]                     851
[Skin, Subcutis and Appendages]                432
Name: count, dtype: int64

In [4]:
df_HE = subset_df_list(df_HE, "T_category", "Blood, Bone Marrow and Lymphatic System")
df_HE = df_HE[df_HE['M_category'].apply(len) == 1]
print(len(df_HE))

2423


In [5]:
# Feature Results
feature_result = r"D:\NOTEBOOKS\Christine\all_slides\feature_summary_old.csv"
df_feature_result = pd.read_csv(feature_result)
df_feature_result = df_feature_result[df_feature_result['status'] == "feature extraction complete"].copy()
df_features = df_feature_result[df_feature_result['model'] == "conch"].copy()
with_features = set(df_features["wsi_path"])
print(len(with_features))

df_HE = df_HE[df_HE["filename"].isin(with_features)]
print(len(df_HE))

2049
701


In [6]:
df_HE = df_HE.drop_duplicates(subset="rekvnr").head(10)
print(df_HE["rekvnr"].value_counts())

rekvnr
13206483    1
13206749    1
12205396    1
13201955    1
13201957    1
13202092    1
13202098    1
12203703    1
12203779    1
13210713    1
Name: count, dtype: int64


In [7]:
all_filenames = df_HE["filename"].tolist()
print("Number of wsi filenames: ", len(all_filenames))

Number of wsi filenames:  10


In [8]:
# Save to csv
output_file = "D:\DATA\spatial_domain_exp1_lymph.csv"

df_HE.to_csv(output_file, index=False)
print(f"Saved DataFrame to {output_file}")

Saved DataFrame to D:\DATA\spatial_domain_exp1_lymph.csv
